In [ ]:
import dynamiqs as dq
dq.set_precision('double')
import numpy as np
from scipy.sparse.linalg import eigs
import scipy.sparse as sp
import scipy.sparse.linalg as spla

In [ ]:
# --- constants ---
ħ = 1
e = 1
phi_quantum = 1 / 2

w_a = 25.338776456203686
kappa_b = 5 / 10.4*2
E_J = 37*2*np.pi /2
phi_a = 0.11
phi_b = 0.204
epsilon_p = 0.02
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b 
kappa_2 = 4*g**2/kappa_b
kappa_1 = 0.005*kappa_2

In [ ]:
n_a = 20

# --- operators ---
a = dq.destroy(n_a)
Ia = dq.eye(n_a)

# pure dissipative cat stabilization
H = 0 * Ia
jump_op_1 = np.sqrt(kappa_1)*a

# --- convert to numpy ---
I_np = np.eye(n_a, dtype=np.complex128)
L_np_1 = jump_op_1.to_numpy()
H_np = H.to_numpy()

LdL_1 = L_np_1.conj().T @ L_np_1

In [ ]:
alpha_sq_list = np.linspace(1,4,10)
bit_flip_rate_list = []
for alpha_sq in alpha_sq_list:
    print(alpha_sq)
    jump_op_2 = np.sqrt(kappa_2) * (a @ a - alpha_sq * Ia)

    # --- convert to numpy ---
    L_np_2 = jump_op_2.to_numpy()
    LdL_2 = L_np_2.conj().T @ L_np_2

    # --- Liouvillian in vectorized form (column stacking) ---
    liouvillian = (
        -1j * (np.kron(I_np, H_np) - np.kron(H_np.T, I_np))
        + np.kron(L_np_2.conj(), L_np_2)
        - 0.5 * np.kron(I_np, LdL_2)
        - 0.5 * np.kron(LdL_2.T, I_np)
        + np.kron(L_np_1.conj(), L_np_1)
        - 0.5 * np.kron(I_np, LdL_1)
        - 0.5 * np.kron(LdL_1.T, I_np)
    )

    vals = np.linalg.eigvals(liouvillian).real
    bit_flip_rate_list.append(-np.sort(vals)[-2])


In [ ]:
bit_flip_rate_list_arnoldi = []
for alpha_sq in alpha_sq_list:
    print(alpha_sq)
    jump_op_2 = np.sqrt(kappa_2) * (a @ a - alpha_sq * Ia)

    # --- convert to numpy ---
    L_np_2 = jump_op_2.to_numpy()
    LdL_2 = L_np_2.conj().T @ L_np_2

    # --- Liouvillian in vectorized form (column stacking) ---
    liouvillian = (
        -1j * (np.kron(I_np, H_np) - np.kron(H_np.T, I_np))
        + np.kron(L_np_2.conj(), L_np_2)
        - 0.5 * np.kron(I_np, LdL_2)
        - 0.5 * np.kron(LdL_2.T, I_np)
        + np.kron(L_np_1.conj(), L_np_1)
        - 0.5 * np.kron(I_np, LdL_1)
        - 0.5 * np.kron(LdL_1.T, I_np)
    )

    vals, _ = eigs(liouvillian, k=2, which = "SM")
    vals = vals.real
    bit_flip_rate_list_arnoldi.append(-np.sort(vals)[-2].real)


In [ ]:
pert_th = kappa_1*alpha_sq_list*np.exp(-4*alpha_sq_list)+kappa_1**2/(2*kappa_2)*np.exp(-2*alpha_sq_list)

In [ ]:
n_a = 20
n_b = 11
N = n_a * n_b

# Operators
a = dq.destroy(n_a)
b = dq.destroy(n_b)
Ia = dq.eye(n_a)
Ib = dq.eye(n_b)

a_tot = dq.tensor(a, Ib)
b_tot = dq.tensor(Ia, b)

# Correct nonlinear term
term = dq.tensor(a @ a, dq.dag(b))
non_linear_op = term + dq.dag(term)
drive_op = b_tot + b_tot.dag()

La = np.sqrt(kappa_1) * a_tot
Lb = np.sqrt(kappa_b) * b_tot

# Convert to sparse CSC
La = sp.csc_matrix(La.to_numpy().astype(np.complex128))
Lb = sp.csc_matrix(Lb.to_numpy().astype(np.complex128))
I = sp.identity(N, dtype=np.complex128, format="csc")

def dissipator(L):
    LdL = L.getH() @ L
    return (
        sp.kron(L.conjugate(), L, format="csc")
        - 0.5 * sp.kron(I, LdL, format="csc")
        - 0.5 * sp.kron(LdL.T, I, format="csc")
    )

In [ ]:
rates_list_arnoldi = []
for alpha_sq in alpha_sq_list:
    print(alpha_sq)
    epsilon_d = 2*alpha_sq*g
    H = g * non_linear_op + (epsilon_d / 2) * drive_op
    H = sp.csc_matrix(H.to_numpy().astype(np.complex128))
    
    Lio = (
    -1j * (sp.kron(I, H, format="csc") - sp.kron(H.T, I, format="csc"))
    + dissipator(La)
    + dissipator(Lb)
    )

    vals, vecs = spla.eigs(Lio, k=6, sigma=1e-12, which="LM", tol=1e-10)
    rates_list_arnoldi.append(-np.sort(vals)[-2].real)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(alpha_sq_list, bit_flip_rate_list, label="Exact diag. one-mode", ls="-")
plt.plot(alpha_sq_list, bit_flip_rate_list_arnoldi, label="Arnoldi default param. one-mode", ls="--")
plt.plot(alpha_sq_list, pert_th, label="Perturbative res. kerr cat")
plt.plot(alpha_sq_list, rates_list_arnoldi, label="Arnoldi two-mode", ls=":")
plt.yscale("log")
plt.xlabel(r"$\alpha^2$")
plt.ylabel("Bit flip rate")
plt.title("Bit flip rate (log scale)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
np.pow(16,1/3)